# inplace-param-update composite — cx6: one full SGD training iteration: step() then zero_grad(set_to_none=True)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `inplace-param-update`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "inplace-param-update"
DD_ATOM_IDS = ["inplace-param-update", "zero-grad-set-none"]
DD_SUBTOPICS = ["PyTorch: In-place param update", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

These two atoms are the two POST-backward halves of one PyTorch training iteration:
```
loss.backward()       # accumulates grads into p.grad.
optimizer.step()      # atom A: in-place update of params using p.grad.
optimizer.zero_grad() # atom B: clear p.grad so the next backward() starts fresh.
```

Both atoms touch `self.params` from the optimizer's materialized list. Both are in-place but in different senses:
- **inplace-param-update** mutates `p.data` (the param's storage).
- **zero-grad-set-none** rebinds `p.grad` to `None` (drops the reference; doesn't mutate the underlying tensor storage).

ORDER matters: step BEFORE zero_grad. If you zero first, you've thrown away the gradient you were about to use to update params, and `.step()` silently no-ops because every `p.grad is None`.

**Anatomy.**
```python
class SGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None: continue
            p.data -= self.lr * p.grad         # atom A.

    def zero_grad(self, set_to_none=True):
        for p in self.params:
            if set_to_none:
                p.grad = None                  # atom B.
            else:
                if p.grad is not None:
                    p.grad.zero_()
```

**Why both atoms together.** This is the smallest 'usable' optimizer — a SGD class with `step + zero_grad` is enough to wire into any training loop and get correct (if slow) convergence. Every other optimizer (Adam, RMSProp) is this pattern with a more elaborate step rule.

### Composite Exercise — one full SGD training iteration: step() then zero_grad(set_to_none=True)

**Atoms exercised together**: `inplace-param-update`, `zero-grad-set-none`

Implement `cx6_make_sgd_with_step_and_zero()` — return a class `SGDFull`.

- `SGDFull(params, lr)`:
  - `self.params = list(params); self.lr = lr`
- `SGDFull.step(self)`:
  - For each `p` with non-None `p.grad`: `p.data -= self.lr * p.grad` (atom A).
  - Wrap in `t.no_grad()`.
- `SGDFull.zero_grad(self, set_to_none=True)`:
  - If `set_to_none`: set every `p.grad = None` (atom B).
  - Else: `p.grad.zero_()` in place for every param with a non-None grad.

The test runs a small autograd-based loop: build a tiny linear model, compute a loss, call `backward()`, then `opt.step(); opt.zero_grad()`, and repeat for 5 iterations. It checks that:
- Params end up at the same values as `torch.optim.SGD` would.
- Between iterations, `p.grad` is `None` (proving zero_grad set_to_none worked).
- The loss DECREASES over iterations (proving step actually moves params toward minimum).
- Forgetting `zero_grad` would have caused gradient accumulation — we verify by NOT calling it and showing the params diverge from the reference.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx6_make_sgd_with_step_and_zero():
    """Return the SGDFull class with both step() and zero_grad()."""
    raise NotImplementedError

def _test_cx6():
    SGDFull = cx6_make_sgd_with_step_and_zero()
    assert isinstance(SGDFull, type)

    # Case A: basic correctness — single step matches torch.optim.SGD, zero_grad clears.
    t.manual_seed(0)
    p_mine = t.nn.Parameter(t.tensor([1.0, 2.0, 3.0]))
    p_ref  = t.nn.Parameter(t.tensor([1.0, 2.0, 3.0]))
    p_mine.grad = t.tensor([0.5, 0.5, 0.5])
    p_ref.grad  = t.tensor([0.5, 0.5, 0.5])
    opt_mine = SGDFull([p_mine], lr=0.1)
    opt_ref  = t.optim.SGD([p_ref], lr=0.1, momentum=0)
    opt_mine.step(); opt_mine.zero_grad()
    opt_ref.step();  opt_ref.zero_grad()
    assert t.allclose(p_mine.data, p_ref.data, atol=1e-7), (
        f'after step+zero_grad, diverges from torch.optim.SGD'
    )
    assert p_mine.grad is None, 'zero_grad() default must set grad to None'
    assert p_ref.grad is None, '(sanity) PyTorch also sets grad to None by default'

    # Case B: full training loop — autograd-driven, 5 iterations, monotone-decreasing loss.
    t.manual_seed(1)
    # Tiny linear-regression-style problem: minimize ||W x - y||^2 over W.
    x = t.randn(8, 3)
    y = t.randn(8, 4)
    W_mine = t.nn.Parameter(t.randn(4, 3))
    W_ref  = t.nn.Parameter(W_mine.detach().clone())
    opt_mine = SGDFull([W_mine], lr=0.05)
    opt_ref  = t.optim.SGD([W_ref], lr=0.05, momentum=0)
    losses = []
    for it in range(5):
        # Mine.
        pred_mine = x @ W_mine.T
        loss_mine = ((pred_mine - y) ** 2).sum()
        loss_mine.backward()
        losses.append(loss_mine.item())
        opt_mine.step()
        opt_mine.zero_grad()
        # Between iterations, grad must be None.
        assert W_mine.grad is None, f'iter {it}: grad should be None after zero_grad()'
        # Reference.
        pred_ref = x @ W_ref.T
        loss_ref = ((pred_ref - y) ** 2).sum()
        loss_ref.backward()
        opt_ref.step()
        opt_ref.zero_grad()
        assert t.allclose(W_mine.data, W_ref.data, atol=1e-6), (
            f'iter {it}: my W diverges from torch.optim.SGD; '
            f'max err = {(W_mine.data - W_ref.data).abs().max().item()}'
        )
    # Loss should be monotonically decreasing (lr small enough for this tiny problem).
    for i in range(1, len(losses)):
        assert losses[i] < losses[i-1], f'loss should decrease, but losses = {losses}'

    # Case C: omitting zero_grad demonstrates accumulation — params would diverge.
    t.manual_seed(2)
    W_noclear = t.nn.Parameter(t.randn(2, 3))
    W_clear   = t.nn.Parameter(W_noclear.detach().clone())
    opt_noclear = SGDFull([W_noclear], lr=0.01)
    opt_clear   = SGDFull([W_clear], lr=0.01)
    x_small = t.randn(4, 3)
    y_small = t.randn(4, 2)
    for it in range(3):
        loss_n = ((x_small @ W_noclear.T - y_small) ** 2).sum(); loss_n.backward()
        opt_noclear.step()   # NO zero_grad — grads accumulate.
        loss_c = ((x_small @ W_clear.T - y_small) ** 2).sum(); loss_c.backward()
        opt_clear.step(); opt_clear.zero_grad()
    # After 3 iterations the no-zero version has accumulated grads → very different params.
    assert not t.allclose(W_noclear.data, W_clear.data, atol=1e-3), (
        'sanity check: omitting zero_grad should cause divergence (gradient accumulation), '
        'but the two trajectories matched — your zero_grad is suspicious'
    )

    # Case D: set_to_none=False clears to zero tensor (not None).
    p4 = t.nn.Parameter(t.randn(3))
    p4.grad = t.ones_like(p4)
    opt4 = SGDFull([p4], lr=0.1)
    opt4.zero_grad(set_to_none=False)
    assert p4.grad is not None and t.all(p4.grad == 0).item(), (
        'zero_grad(set_to_none=False) must leave grad as a zero tensor, not None'
    )
    _dd_passed.add('cx6')

_test_cx6()

<details><summary>Show solution — cx6</summary>

```python
def cx6_make_sgd_with_step_and_zero():
    class SGDFull:
        def __init__(self, params, lr):
            self.params = list(params)
            self.lr = lr

        @t.no_grad()
        def step(self):
            for p in self.params:
                if p.grad is None:
                    continue
                # Atom A (inplace-param-update): p.data -= lr * p.grad.
                p.data -= self.lr * p.grad

        def zero_grad(self, set_to_none=True):
            for p in self.params:
                if set_to_none:
                    # Atom B (zero-grad-set-none): drop the reference.
                    p.grad = None
                else:
                    if p.grad is not None:
                        p.grad.zero_()

    return SGDFull
```

The two atoms are intentionally symmetric in shape but different in semantics: `step` mutates `p.data` (tensor storage), `zero_grad(set_to_none=True)` rebinds `p.grad` (reference). The training-loop ORDER `step → zero_grad` is what every PyTorch tutorial shows; swap them and step silently no-ops. The `@t.no_grad()` on `step` is the autograd-side guarantee that the optimizer's own arithmetic doesn't build a graph through `p.data` (which would be silently ignored anyway, but builds garbage for the gc to clean).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx6',
        'subtopics': ["PyTorch: In-place param update", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()